In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv('../data/dataset.csv')

# Convert EUI to numbers ("Not Available" becomes NaN)
df['EUI'] = pd.to_numeric(df['Site EUI (kBtu/ft²)'], errors='coerce')

# Drop rows with no EUI — can't train without a target
df = df.dropna(subset=['EUI'])

# Remove outliers — values that are physically impossible
df = df[(df['EUI'] > 1) & (df['EUI'] < 1000)]

# Remove future-dated buildings
df = df[df['Year Built'] <= 2025]

print(f"Rows after cleaning: {len(df):,}")

/var/folders/mj/1y9gngxs6v52_t9ksh10c0wh0000gn/T/ipykernel_54705/3546078528.py:7: DtypeWarning: Columns (0: Postal Code, 1: Largest Property Use Type - Gross Floor Area (ft²), 2: Eligible for Certification for Report PED (Y/N), 3: ENERGY STAR Certification - Application Status, 4: Percent Electricity, 5: Natural Gas Use (therms), 6: Total (Location-Based) GHG Emissions (Metric Tons CO2e), 7: Total (Location-Based) GHG Emissions Intensity (kgCO2e/ft²), 8: Direct GHG Emissions (Metric Tons CO2e), 9: Direct GHG Emissions Intensity (kgCO2e/ft²), 10: Indirect (Location-Based) GHG Emissions (Metric Tons CO2e), 11: Indirect (Location-Based) GHG Emissions Intensity (kgCO2e/ft²), 12: Net Emissions (Metric Tons CO2e), 13: Default Values, 14: Institutional Property? (Y/N), 15: Data Quality Checker Run?, 16: Data Quality Checker - Date Run, 17: Last Modified Date - Property, 18: Last Modified Date - Electric Meters, 19: Electric Distribution Utility, 20: Last Modified Date - Gas Meters, 21: Last M

Rows after cleaning: 89,397


In [4]:
df['sqft'] = pd.to_numeric(df['Largest Property Use Type - Gross Floor Area (ft²)'], errors='coerce')
df['year_built'] = pd.to_numeric(df['Year Built'], errors='coerce')
df['num_buildings'] = pd.to_numeric(df['Number of Buildings'], errors='coerce')

features = pd.get_dummies(df['Primary Property Type - Self Selected'], prefix='type')

features['sqft'] = df['sqft']
features['year_built'] = df['year_built']
features['num_buildings'] = df['num_buildings']

# Drop rows with any missing feature values
features['EUI'] = df['EUI']
features = features.dropna()

# Separate inputs (X) from target (y)
y = features['EUI']
X = features.drop(columns=['EUI'])

print(f"Rows: {len(X):,}")
print(f"Features: {X.shape[1]}")
print(f"First few column names: {list(X.columns[:5])}")

Rows: 89,397
Features: 83
First few column names: ['type_Adult Education', 'type_Ambulatory Surgical Center', 'type_Aquarium', 'type_Automobile Dealership', 'type_Bank Branch']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training EUI mean: {y_train.mean():.1f}, median: {y_train.median():.1f}")
print(f"Test EUI mean:     {y_test.mean():.1f}, median: {y_test.median():.1f}")

Training EUI mean: 78.8, median: 71.5
Test EUI mean:     78.5, median: 71.4
